#### CAPM回归
经过2_get_label与3_make_25per，现在有：
* month_label包含全部基金的月收益率label和基金的权重weight
* top25_month_label包含每个月top25%的基金的月收益率label和基金的权重weight
* month_HS300_G1Y包含沪深300（大盘）和一年期国债利率（无风险利率）的月收益率

In [11]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from datetime import datetime

# pd.set_option('display.float_format', '{:.4f}'.format)  # 4位小数

In [12]:
fund_ret = pd.read_feather('data/month_label.feather')
fund_25_ret = pd.read_feather('data/top25_month_label.feather')
index_ret = pd.read_feather('data/month_HS300_rf.feather')
print(fund_ret.head())
print(fund_25_ret.head())
print(index_ret.head())

                      nav_date  accum_nav  adj_nav     fd_share     label  \
ann_date   ts_code                                                          
2016-04-30 000011.OF  20160429     13.937  17.2133   18642.5278  0.002014   
           000309.OF  20160429      1.539   1.5390   32296.7705  0.035666   
           000409.OF  20160429      1.401   1.4010   36018.4498  0.007914   
           000471.OF  20160429      2.262   2.2620  177241.4719 -0.013089   
           000513.OF  20160429      1.390   1.3900   74609.3099 -0.024561   

                        weight  
ann_date   ts_code              
2016-04-30 000011.OF  0.000767  
           000309.OF  0.001328  
           000409.OF  0.001481  
           000471.OF  0.007290  
           000513.OF  0.003069  
                      adj_nav     label    weight
ann_date   ts_code                               
2017-04-30 000309.OF    1.836 -0.024442  0.022620
           000409.OF    1.670  0.006024  0.007996
           000577.OF    2.384

In [13]:
ff3 = pd.read_feather('ff3/ff3_factors.feather')
ff3

,SMB,HML,mkt_rf,rf
ann_date,,,,
2016-04-30,-0.001156,0.000638,0.000912,0.000041
2016-05-31,0.001350,-0.011931,-0.014808,0.000041
2016-06-30,0.004969,0.007979,-0.026001,0.000041
2016-07-31,-0.001856,0.001856,-0.003304,0.000041
2016-08-31,0.000117,0.001372,0.003461,0.000041
...,...,...,...,...
2025-11-30,0.009435,-0.003411,0.004468,0.000041
2025-12-31,0.002131,0.004884,0.003474,0.000041
2026-01-31,0.010251,-0.000439,0.008118,0.000041


In [16]:
all_ret = fund_ret.groupby('ann_date')['label'].mean()
fund_excess = all_ret - index_ret['yield']
fund_excess

ann_date
2016-04-30   -0.016587
2016-05-31   -0.041108
2016-06-30    0.074090
2016-07-31   -0.008056
2016-08-31    0.014593
                ...   
2025-11-30   -0.044126
2025-12-31    0.031220
2026-01-31    0.050877
2026-02-28    0.003684
2026-03-31   -0.071253
Length: 120, dtype: float64

In [14]:
market_excess = index_ret['excess']
smb = ff3['SMB']
hml = ff3['HML']

In [17]:
# 全部基金等权收益率的超额收益回归
X = pd.DataFrame({
    'market': market_excess,
    'smb': smb,
    'hml': hml
})

# 添加截距项
X = sm.add_constant(X)
y = fund_excess

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.610
Model:                            OLS   Adj. R-squared:                  0.600
Method:                 Least Squares   F-statistic:                     60.51
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           1.28e-23
Time:                        12:22:06   Log-Likelihood:                 252.13
No. Observations:                 120   AIC:                            -496.3
Df Residuals:                     116   BIC:                            -485.1
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0036      0.003      1.277      0.2

const的系数alpha仅为0.0034，不支持“整体基金跑赢市场的结论”

进一步可以做：
* 使用滚动回归计算每只基金的 CAPM alpha（过去12周） -> 按 alpha 排序，取前25%构建等权组合。-> 计算 Top 组合的后续收益和 alpha（仍用 CAPM 评估）。 -> 如果 Top 组合的 alpha 显著为正，且显著高于全体组合，则说明存在持续性。
* 设置无风险利率为0，看alpha
* 增大滚动窗口，改用2周～一个月等，但alpha估计不稳定，谨慎！

In [18]:
fund_weighted_excess = fund_ret.groupby('ann_date').apply(lambda x: np.sum(x['label'] * x['weight'])) - index_ret['yield']

In [20]:
# 全部基金加权收益率的超额收益回归
X = pd.DataFrame({
    'market': market_excess,
    'smb': smb,
    'hml': hml
})
# 添加截距项
X = sm.add_constant(X)
y = fund_weighted_excess

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.626
Model:                            OLS   Adj. R-squared:                  0.616
Method:                 Least Squares   F-statistic:                     64.73
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           1.16e-24
Time:                        12:22:45   Log-Likelihood:                 255.76
No. Observations:                 120   AIC:                            -503.5
Df Residuals:                     116   BIC:                            -492.4
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0020      0.003      0.731      0.4

In [21]:
all_25_ret = fund_25_ret.groupby('ann_date')['label'].mean()
fund_25_excess = all_25_ret - index_ret['yield']
fund_25_excess

2016-04-30         NaN
2016-05-31         NaN
2016-06-30         NaN
2016-07-31         NaN
2016-08-31         NaN
                ...   
2025-11-30   -0.055362
2025-12-31    0.069711
2026-01-31    0.049054
2026-02-28    0.016276
2026-03-31   -0.067945
Length: 120, dtype: float64

In [23]:
# 全部基金加权收益率的超额收益回归
X = pd.DataFrame({
    'market': market_excess,
    'smb': smb,
    'hml': hml
})
# 添加截距项
X = sm.add_constant(X)
y = fund_25_excess

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.538
Model:                            OLS   Adj. R-squared:                  0.525
Method:                 Least Squares   F-statistic:                     40.44
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           2.10e-17
Time:                        12:23:00   Log-Likelihood:                 204.20
No. Observations:                 108   AIC:                            -400.4
Df Residuals:                     104   BIC:                            -389.7
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0080      0.004      2.154      0.0

In [24]:
# fund_weighted_excess = fund_ret.groupby('ann_date').apply(lambda x: np.sum(x['label'] * x['weight'])) - index_ret['yield']
fund_25weighted_excess = fund_25_ret.groupby('ann_date').apply(lambda x: np.sum(x['label'] * x['weight'])) - index_ret['yield']

In [25]:
# 全部基金加权收益率的超额收益回归
X = pd.DataFrame({
    'market': market_excess,
    'smb': smb,
    'hml': hml
})
# 添加截距项
X = sm.add_constant(X)
y = fund_25weighted_excess

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.574
Model:                            OLS   Adj. R-squared:                  0.561
Method:                 Least Squares   F-statistic:                     46.64
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           3.51e-19
Time:                        12:23:15   Log-Likelihood:                 204.72
No. Observations:                 108   AIC:                            -401.4
Df Residuals:                     104   BIC:                            -390.7
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0078      0.004      2.120      0.0